In [ ]:
from syft_rds.orchestra import setup_rds_stack
from rds_chat_analysis import REPO_ROOT
import pandas as pd
from rds_chat_analysis import DATA_DIR
import datetime

In [ ]:
key = "wildchat"
stack = setup_rds_stack(
    root_dir=REPO_ROOT / ".rds",
    key=key,
    log_level="DEBUG",
    reset=False,
)

do_client = stack.do_rds_client
ds_client = stack.ds_rds_client

In [ ]:
wildchat_dataset = ds_client.datasets[0]
wildchat_dataset.describe()

In [ ]:
mock_data = pd.read_parquet(
    wildchat_dataset.mock_path / "data.parquet",
)

mock_data

In [ ]:
from rds_chat_analysis.job_utils import prepare_job_dependencies

submission_dir = (
    DATA_DIR / "jobs" / datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
)
submission_dir.mkdir(parents=True, exist_ok=True)
job_entrypoint_path = submission_dir / "main.py"

prepare_job_dependencies(submission_dir)

print("Prepared job submission in ", submission_dir)
print("Please add your `main.py` file and submit the job to the RDS server")

In [ ]:
%%writefile {job_entrypoint_path}

from pathlib import Path
import os

DATA_DIR = os.environ["DATA_DIR"]
OUTPUT_DIR = os.environ["OUTPUT_DIR"]
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

with open(Path(OUTPUT_DIR) / "out.txt", "w") as f:
    f.write("Hello from the job!")
    try:
        print("Testing job submission...")

        from rds_chat_analysis.job_utils import test_job_submission
        f.write(test_job_submission())

    except Exception as e:
        print(f"Error: {e}", file=f)


In [ ]:
ds_client.jobs.submit(
    user_code_path=submission_dir,
    dataset_name=wildchat_dataset.name,
    entrypoint="main.py",
)

In [ ]:
ds_client.jobs.get_all()[0].user_code.describe()

In [ ]:
job = do_client.jobs.get_all()[0]

In [ ]:
job.describe()

In [ ]:
result = do_client.run_private(job)

In [ ]:
print(result.output_path)

In [ ]:
do_client.jobs.share_results(result)

In [ ]:
(ds_client.jobs.get_all()[0].output_path / "output" / "out.txt").read_text()

In [ ]:
job.describe()